# PPVMD 整理版工作流模板

本 notebook 使用重新分类整理后的 `ppvmd_tools` 包。先把 `ppvmd_tools` 文件夹放在当前 notebook 同级目录，或把 zip 解压到当前工作目录。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from ppvmd_tools.ppvmd_denoising import physiology_preserving_adaptive_denoise
from ppvmd_tools.ppvmd_validation import (
    recommend_adaptive_parameters,
    make_semisynthetic_pressure_dataset,
    evaluate_semisynthetic_result,
)
from ppvmd_tools.ppvmd_vmd import (
    physiology_constrained_vmd_reconstruction,
    summarize_physiology_vmd_result,
    search_best_vmd_parameters,
)

OUTPUT_DIR = Path("paper_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. 数据读取

In [2]:
RAW_DATA_PATH = r"D:\a_work\课题组实验数据处理\电子胶囊临床实验数据\导出数据\2bHT000007.txt"

TIME_COL = "Time"
PH_COL = "PH"
TEMP_COL = "Temperature"
PRESSURE_COL = "Pressure"

PRESSURE_MIN = 85
PRESSURE_MAX = 130
RESAMPLE_DT = 1.2

data_raw = pd.read_csv(
    RAW_DATA_PATH,
    delimiter="\t",
    header=0,
    names=[TIME_COL, PH_COL, TEMP_COL, PRESSURE_COL],
).reset_index(drop=True)

for col in [TIME_COL, PH_COL, TEMP_COL, PRESSURE_COL]:
    data_raw[col] = pd.to_numeric(data_raw[col], errors="coerce")

data_raw = data_raw.dropna(subset=[TIME_COL, PH_COL, TEMP_COL, PRESSURE_COL]).reset_index(drop=True)

colon_data = data_raw.iloc[25000:40900].copy().reset_index(drop=True)
colon_data.head()

,Time,PH,Temperature,Pressure
0,39809.23,8.882,36.549,101.123
1,39810.38,8.265,36.430,101.817
2,39811.52,8.333,36.549,101.865
3,39812.66,8.333,36.537,100.901
4,39813.81,9.088,36.430,100.496


## 2. 自适应预处理降噪

In [3]:
adaptive_result = recommend_adaptive_parameters(
    colon_data,
    time_col=TIME_COL,
    pressure_col=PRESSURE_COL,
    pressure_min=PRESSURE_MIN,
    pressure_max=PRESSURE_MAX,
)

allowed_keys = [
    "noise_window_s", "event_amp_k", "event_area_k",
    "amp_k", "slope_k", "curvature_k", "smooth_window_s",
]
adaptive_kwargs = {k: adaptive_result[k] for k in allowed_keys if k in adaptive_result}
quality_metrics = {k: v for k, v in adaptive_result.items() if k not in allowed_keys}

print("自适应参数:", adaptive_kwargs)
print("信号质量:", quality_metrics)

df_out = physiology_preserving_adaptive_denoise(
    colon_data,
    time_col=TIME_COL,
    pressure_col=PRESSURE_COL,
    ph_col=PH_COL,
    temp_col=TEMP_COL,
    resample_dt=RESAMPLE_DT,
    use_hard_range_gate=True,
    pressure_min=PRESSURE_MIN,
    pressure_max=PRESSURE_MAX,
    **adaptive_kwargs,
)

df_out.head()

自适应参数: {'noise_window_s': 22.233803187973674, 'event_amp_k': 3.1787042550378937, 'event_area_k': 1.6116901593986837, 'amp_k': 6.164929521803949, 'slope_k': 6.164929521803949, 'curvature_k': 6.164929521803949, 'smooth_window_s': 5.470140956392102}
信号质量: {'signal_quality_badness': 0.2233803187973673, 'range_abnormal_ratio': 0.0003773584905660377, 'noise_ratio': 0.34192037434718336}


,Time,PH,Temperature,Pressure_Raw,Pressure_RangeFixed,Pressure_Preclean,Pressure_Clean,Baseline,Residual,LocalNoiseSigma,RangeAbnormalMask,ArtifactMask,PhysioEventMask,QualityFlag
0,39809.23,8.882000,36.549000,101.123000,101.123000,101.123000,101.237362,101.102000,0.135362,0.301927,False,False,False,good
1,39810.43,8.267982,36.435219,101.819105,101.819105,101.819105,101.637869,101.070447,0.567422,0.272665,False,False,False,good
2,39811.63,8.333000,36.547842,101.771982,101.771982,101.771982,101.629520,101.098816,0.530705,0.275125,False,False,False,good
3,39812.83,8.444609,36.521183,100.841130,100.841130,100.841130,100.975415,101.064122,-0.088707,0.276055,False,False,False,good
4,39814.03,9.088000,36.434632,100.548105,100.548105,100.548105,100.595274,101.042368,-0.447095,0.261197,False,False,False,good


## 3. 半合成验证

In [4]:
semi_source = df_out[["Time", "PH", "Temperature", "Baseline", "PhysioEventMask"]].copy()
semi_source["Pressure"] = df_out["Pressure_Clean"].values

semi_df, truth = make_semisynthetic_pressure_dataset(
    semi_source,
    time_col="Time",
    pressure_col="Pressure",
    ph_col="PH",
    temp_col="Temperature",
    event_col="PhysioEventMask",
)

adaptive_result_semi = recommend_adaptive_parameters(
    semi_df, time_col="Time", pressure_col="Pressure",
    pressure_min=PRESSURE_MIN, pressure_max=PRESSURE_MAX,
)
adaptive_kwargs_semi = {k: adaptive_result_semi[k] for k in allowed_keys if k in adaptive_result_semi}

semi_pred_df = physiology_preserving_adaptive_denoise(
    semi_df,
    time_col="Time", pressure_col="Pressure", ph_col="PH", temp_col="Temperature",
    resample_dt=RESAMPLE_DT,
    use_hard_range_gate=True, pressure_min=PRESSURE_MIN, pressure_max=PRESSURE_MAX,
    **adaptive_kwargs_semi,
)

semi_report = evaluate_semisynthetic_result(
    semi_pred_df, truth,
    time_col="Time", clean_col="Pressure_Clean", artifact_col="ArtifactMask", event_col="PhysioEventMask",
)
semi_report["summary"]

,RMSE,Correlation,Artifact_F1,Artifact_Recall,Mean_Event_IoU,Mean_Abs_OnsetError_s,Mean_Abs_OffsetError_s
0,0.081321,0.998942,0.504762,0.634731,0.726398,2.976623,1.932468


## 4. 生理约束型 VMD

In [5]:
MIN_KEEP_SCORE = 0.18
MAIN_K = 6
MAIN_ALPHA = 1500

main_vmd = physiology_constrained_vmd_reconstruction(
    signal=df_out["Pressure_Clean"].values,
    time=df_out["Time"].values,
    event_mask=df_out["PhysioEventMask"].values,
    artifact_mask=df_out["ArtifactMask"].values,
    K=MAIN_K,
    alpha=MAIN_ALPHA,
    min_keep_score=MIN_KEEP_SCORE,
)

main_mode_summary = summarize_physiology_vmd_result(main_vmd)
main_mode_summary

,Mode,DominantFreq_Hz,EnergyRatio,SignalCorrelation,EventCorrelation,EventEnergyRatio,ArtifactEnergyRatio,SpectralEntropy,Kurtosis,TemporalContinuity,PhysioKeepScore,Selected
0,1,0.000052,0.688252,0.857647,0.486943,0.159359,0.098164,0.277281,5.740569,0.999950,0.457827,True
1,2,0.006270,0.082164,0.399256,0.312161,0.346086,0.118364,0.578818,17.861682,0.993887,0.253855,True
2,3,0.021971,0.060964,0.330154,0.747192,0.394547,0.144214,0.601091,22.232414,0.966194,0.328568,True
3,4,0.054512,0.023132,0.222102,0.408689,0.270035,0.140441,0.660830,19.971495,0.866716,0.179976,False
4,5,0.087727,0.018229,0.190814,0.296509,0.190591,0.086568,0.668506,14.015759,0.705020,0.145395,False
5,6,0.119803,0.008022,0.144064,0.202567,0.172900,0.065199,0.707432,10.158976,0.547397,0.103553,False


## 5. K-alpha 参数搜索

In [6]:
K_list = [3, 4, 5, 6, 7]
alpha_list = [1500, 2000, 2500, 2700, 3000, 3500]

vmd_param_search_result = search_best_vmd_parameters(
    pred_df=df_out,
    signal_col="Pressure_Clean",
    K_list=K_list,
    alpha_list=alpha_list,
    min_keep_score=MIN_KEEP_SCORE,
)

vmd_param_search_result.head(10)

Done: K=3, alpha=1500, Score=0.8958, RMSE=0.4590, Corr=0.9667, Modes=[1, 2]
Done: K=3, alpha=2000, Score=0.8885, RMSE=0.4723, Corr=0.9647, Modes=[1, 2]
Done: K=3, alpha=2500, Score=0.8820, RMSE=0.4835, Corr=0.9631, Modes=[1, 2]
Done: K=3, alpha=2700, Score=0.8798, RMSE=0.4875, Corr=0.9625, Modes=[1, 2]
Done: K=3, alpha=3000, Score=0.8762, RMSE=0.4933, Corr=0.9616, Modes=[1, 2]
Done: K=3, alpha=3500, Score=0.8714, RMSE=0.5023, Corr=0.9602, Modes=[1, 2]
Done: K=4, alpha=1500, Score=0.9335, RMSE=0.3767, Corr=0.9777, Modes=[1, 2, 3]
Done: K=4, alpha=2000, Score=0.9271, RMSE=0.3882, Corr=0.9764, Modes=[1, 2, 3]
Done: K=4, alpha=2500, Score=0.9215, RMSE=0.3980, Corr=0.9752, Modes=[1, 2, 3]
Done: K=4, alpha=2700, Score=0.9191, RMSE=0.4016, Corr=0.9748, Modes=[1, 2, 3]
Done: K=4, alpha=3000, Score=0.9157, RMSE=0.4069, Corr=0.9742, Modes=[1, 2, 3]
Done: K=4, alpha=3500, Score=0.9111, RMSE=0.4152, Corr=0.9731, Modes=[1, 2, 3]
Done: K=5, alpha=1500, Score=0.9283, RMSE=0.3969, Corr=0.9752, Modes=[

,K,alpha,min_keep_score,RMSE,RMSE_Norm,Correlation,Mean_Event_IoU,Mean_PeakPreservationRatio,Mean_AreaPreservationRatio,NumEvents,NumSelectedModes,SelectedModes,ObjectiveScore
0,7,1500,0.18,0.390937,0.023043,0.975773,0.994369,0.793221,0.876477,78,4,"[1, 2, 3, 4]",0.937511
1,6,2000,0.18,0.386330,0.022772,0.976384,0.994369,0.789585,0.872675,78,4,"[1, 2, 3, 4]",0.936616
2,7,2000,0.18,0.395074,0.023287,0.975277,0.994369,0.783339,0.866504,78,4,"[1, 2, 3, 4]",0.934348
3,4,1500,0.18,0.376659,0.022202,0.977706,0.993453,0.780802,0.858856,78,3,"[1, 2, 3]",0.933515
4,6,2500,0.18,0.391362,0.023068,0.975783,0.994369,0.779539,0.862655,78,4,"[1, 2, 3, 4]",0.933382
5,6,2700,0.18,0.393225,0.023178,0.975560,0.994369,0.775809,0.858876,78,4,"[1, 2, 3, 4]",0.932172
6,7,2500,0.18,0.399043,0.023521,0.974799,0.994369,0.774158,0.857153,78,4,"[1, 2, 3, 4]",0.931390
7,6,3000,0.18,0.395903,0.023336,0.975238,0.994369,0.770484,0.853407,78,4,"[1, 2, 3, 4]",0.930433
8,7,2700,0.18,0.400612,0.023614,0.974608,0.994369,0.770614,0.853547,78,4,"[1, 2, 3, 4]",0.930247
9,7,3000,0.18,0.402978,0.023753,0.974320,0.994369,0.765398,0.848239,78,4,"[1, 2, 3, 4]",0.928561


## 6. 结果导出

In [7]:
df_out.to_csv(OUTPUT_DIR / "denoised_signal.csv", index=False)
main_mode_summary.to_csv(OUTPUT_DIR / "final_vmd_mode_summary.csv", index=False)
vmd_param_search_result.to_csv(OUTPUT_DIR / "vmd_parameter_search_result.csv", index=False)
print("Exported to", OUTPUT_DIR.resolve())

Exported to D:\a_work\课题组实验数据处理\新预处理\paper_outputs
